# marv-hyena on Colab: looking inside Evo 2

This notebook loads **Evo 2 7B** and uses [marv-hyena](https://github.com/thebnbrkr/marv-hyena) to find out
which part of the model does which job:

| type | how far it looks | Evo 2 7B blocks |
|---|---|---|
| **SE**: Hyena short explicit | 7 letters | 0 4 7 11 14 18 21 25 28 |
| **MR**: Hyena medium regularized | 128 letters | 1 5 8 12 15 19 22 26 29 |
| **LI**: Hyena long implicit | whole sequence, fading | 2 6 9 13 16 20 23 27 30 |
| **attn**: attention | whole sequence, exact lookup | 3 10 17 24 31 |

**Runtime:** `Runtime → Change runtime type → A100 GPU`, with **High-RAM** on.
An L4 may work with shorter sequences. A T4 will **not** work, because it has no bfloat16 support.
No flash-attn install is needed; see the setup cell.

**Order:** set up, run the smoke checks (they must pass), then the experiments. Each experiment tests one prediction in
[`PREDICTIONS.md`](https://github.com/thebnbrkr/marv-hyena/blob/main/PREDICTIONS.md).

## 0. Setup

In [ ]:
# GPU check (no torch import yet: the install below may change the torch version)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# OPTIONAL: keep the ~15 GB Evo 2 weights on Google Drive so the next session skips the download.
# This must run BEFORE anything imports huggingface_hub / evo2.
USE_DRIVE = False

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF_HOME =', os.environ.get('HF_HOME', '(default ~/.cache/huggingface)'))

In [ ]:
# Install Evo 2 + marv-hyena (~2-5 min). NO flash-attn: pip usually finds no prebuilt wheel for Colab's torch
# and compiles for hours. marv-hyena instead runs Evo 2's attention through PyTorch's built-in fused kernel
# (scaled_dot_product_attention), which is also fast on an A100 -- see marv_hyena/noflash.py.
# Colab runs Python 3.13, but every current evo2 release declares Python < 3.13, so a plain `pip install evo2`
# silently falls back to the old, incompatible 0.3.0. evo2 is pure Python, so force the current release.
# Colab also ships an empty `transformer-engine` package whose import crashes evo2; remove it (7B needs no TE).
!pip uninstall -y -q transformer-engine transformer_engine
!pip install -q --ignore-requires-python evo2==0.6.0
!rm -rf marv-hyena && git clone -q https://github.com/thebnbrkr/marv-hyena.git
!pip install -q -r marv-hyena/requirements.txt openpyxl

import sys
sys.path.insert(0, '/content/marv-hyena')

In [ ]:
# Must run before anything imports evo2/vortex: lets Vortex import without flash-attn and turns flash attention off.
from marv_hyena import noflash
noflash.prepare()

import torch, vortex, evo2
from importlib.metadata import version
print('evo2', version('evo2'), '| vtx', version('vtx'))
assert version('evo2') >= '0.6.0', 'old evo2 installed: Runtime -> Disconnect and delete runtime, then rerun from the top'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| flash-attn package installed:', noflash.flash_attn_available())
print('GPU', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0)[0] >= 8, 'needs an Ampere+ GPU with bf16 (A100 / L4 / H100)'

In [ ]:
# marv-hyena's own tests (a tiny CPU model that mirrors Vortex). Should say "35 passed".
!cd marv-hyena && python -m pytest -q

In [ ]:
# Data: the annotated E. coli K-12 genome from the evo2 repo
import os, urllib.request
os.makedirs('data', exist_ok=True)
EVO2_RAW = 'https://raw.githubusercontent.com/ArcInstitute/evo2/main/notebooks'
GENOME = 'data/NC_000913.gb'
if not os.path.exists(GENOME):
    urllib.request.urlretrieve(f'{EVO2_RAW}/sparse_autoencoder/NC_000913.gb', GENOME)

import marv_hyena as mh
from marv_hyena.probes import load_sequence
genome = load_sequence(GENOME)
print(f'E. coli genome: {len(genome):,} letters')

In [ ]:
# Load Evo 2 7B (downloads ~15 GB the first time)
import time
MODEL_NAME = 'evo2_7b'
t0 = time.time()
hm = mh.HyenaModel.load(MODEL_NAME)
print(f'loaded in {time.time()-t0:.0f}s')
print(hm.describe())
RESULTS = {}

## 1. Smoke checks: must pass before anything below is trusted

These check every assumption marv-hyena makes about the installed Vortex: every residual write is captured, the trace
adds up to the real logits, the distance bands rebuild each Hyena block's real output, the block-0 receptive field, and
that hooks leave no trace. If one fails, **stop**: the numbers below would mean nothing.

In [ ]:
from marv_hyena.checks import run_smoke_checks
seq = genome[100_000:104_096]
ok = run_smoke_checks(hm, seq)
assert ok, 'smoke checks failed: do not trust later results; please report the output above'

## 2. Why did the model predict this letter? (trace + distance)

`decompose_prediction` splits the model's preference for the right letter (its logit minus the mean of the other three
bases) **exactly** into what each block wrote. These are direct effects, grouped by operator type.

In [ ]:
import matplotlib.pyplot as plt

INDEX = 3000                       # a letter inside seq to explain
d = mh.decompose_prediction(hm, seq, INDEX)
d.show(k=12)

agg = d.by_kind()
keys = [k for k in ['embed', 'se', 'mr', 'li', 'attn', 'mlp'] if k in agg]
plt.figure(figsize=(6, 3))
plt.bar(keys, [agg[k] for k in keys], color=['grey', 'C0', 'C1', 'C2', 'C3', 'C4'][:len(keys)])
plt.axhline(0, color='k', lw=0.5)
plt.title(f'direct contribution to predicting letter {INDEX} ({seq[INDEX]})')
plt.ylabel('logit difference'); plt.show()

In [ ]:
# How far back was each Hyena block looking when it made that push?
# Each cell = direct logit contribution coming from letters that many positions back (exact split).
import numpy as np
direction = mh.normed_direction(hm, seq, INDEX)
prof = mh.distance_profile(hm, seq[:INDEX], INDEX - 1)
mh.show_profile(prof, direction=direction)

M = np.array([r.project(direction) for r in prof])
lim = np.abs(M).max()
plt.figure(figsize=(7, 7))
plt.imshow(M, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
plt.yticks(range(len(prof)), [f'L{r.block} {r.kind}' for r in prof])
plt.xticks(range(len(prof[0].bands)), [mh.distance.band_label(b) for b in prof[0].bands])
plt.xlabel('letters back'); plt.colorbar(label='direct logit contribution'); plt.show()

## 3. How far does each Hyena block look, from the weights alone? (P4)

No forward pass. SE and MR filters are read directly. LI filters are sums of decaying exponentials, evaluated out to
1M letters. `reach99` = the distance holding 99% of the filter's weight (median over channels).

In [ ]:
reach = mh.reach_table(hm)
mh.show_reach(reach)
RESULTS['reach'] = [{'block': r.block, 'kind': r.kind, 'reach50': r.reach50, 'reach90': r.reach90,
                     'reach99': r.reach99, 'max_reach99': r.max_reach99} for r in reach]

colors = {'se': 'C0', 'mr': 'C1', 'li': 'C2'}
plt.figure(figsize=(9, 3))
plt.bar([str(r.block) for r in reach], [max(r.reach99, 1) for r in reach], color=[colors[r.kind] for r in reach])
plt.yscale('log'); plt.xlabel('block'); plt.ylabel('reach99 (letters, log)')
plt.title('weight-read reach: SE blue, MR orange, LI green'); plt.show()

## 4. Who copies across long distances? (P1)

A random 200-letter insert appears twice inside real E. coli DNA, `gap` letters apart. The first copy can't be
predicted; the second can only be predicted by finding and copying the first. Each operator type's mixers are
mean-ablated in turn.

**Prediction P1:** `-attn` kills copying at gaps ≥ 1,000; `-li` barely matters. Add `50000` to `GAPS` if memory allows.

In [ ]:
from marv_hyena.experiments import copy_test, summarize_copy
GAPS = (100, 1000, 10000)
copy_rows = copy_test(hm, genome, gaps=GAPS, insert_len=200, seeds=2)
RESULTS['copy_test'] = copy_rows

gaps, conds, a2, a1 = summarize_copy(copy_rows)
plt.figure(figsize=(7, 4))
for j, c in enumerate(conds):
    plt.plot(gaps, a2[:, j], marker='o', label=c)
plt.plot(gaps, a1[:, 0], 'k--', label='first copy (chance-ish)')
plt.xscale('log'); plt.xlabel('gap (letters)'); plt.ylabel('second-copy accuracy'); plt.legend(); plt.show()

## 5. Who tracks the reading frame? (P3)

Inside genes, DNA is read in 3-letter codons, and a good model predicts some codon positions better than others.
The table shows accuracy per codon position, with each operator type ablated.

**Prediction P3:** `-se` flattens the 3-letter rhythm most.

In [ ]:
import pandas as pd
from marv_hyena.experiments import codon_test
track = mh.genbank_track(GENOME, 200_000, 208_192)
codon_rows = codon_test(hm, track)
RESULTS['codon_test'] = codon_rows
pd.DataFrame(codon_rows).pivot(index='condition', columns='region', values='acc').round(3)

## 6. Does far-away DNA help, and which operator carries it? (P2)

The first 1,000 letters of a gene are scored with 50,000 letters of upstream DNA, then with only 500.
The difference is the *context benefit*. Which ablation removes it?

**Prediction P2:** `-li` removes ≥ 50% of the benefit; `-attn` removes less.

In [ ]:
from Bio import SeqIO
from marv_hyena.experiments import context_test
rec = next(SeqIO.parse(GENOME, 'genbank'))
cds = next(f for f in rec.features if f.type == 'CDS' and f.location.strand == 1
           and int(f.location.start) > 60_000 and len(f.location) > 1200)
s = int(cds.location.start)
UP = 50_000
window = genome[s - UP: s + 1000]
print('gene:', cds.qualifiers.get('gene', ['?'])[0], 'at', s)
ctx_rows = context_test(hm, window, target=(UP, UP + 1000), short=500)
RESULTS['context_test'] = ctx_rows
df = pd.DataFrame(ctx_rows)
df['benefit_kept'] = df['context_benefit'] / df.loc[df.condition == 'none', 'context_benefit'].item()
df.round(4)

## 7. The first layer's complete motif dictionary (P6)

Block 0 only sees the last 9 letters, so we run **all 4⁹ = 262,144** possible inputs through it: an exact list of
what every channel detects, with nothing sampled. This takes a few minutes. Channels are ranked by motif sharpness
(information content).

In [ ]:
from marv_hyena import motifs
md_ = motifs.enumerate_block0(hm)

def ic(p):
    p = np.clip(p, 1e-9, 1); return float((2 + (p * np.log2(p)).sum(1)).sum())

ics = np.array([ic(md_.pwm(c)) for c in range(hm.hidden_size)])
order = np.argsort(-ics)
print(f'channels with IC >= 8 bits: {(ics >= 8).mean():.1%}')
print(f"{'ch':>5} {'IC':>6} {'consensus':>10}  top k-mers")
for c in order[:25]:
    print(f'{c:>5} {ics[c]:>6.2f} {md_.consensus(c):>10}  {", ".join(md_.top_kmers[c][:3])}')

# Known signals: channels where most of the 50 strongest k-mers contain the motif
for name, motif in [('start ATG', 'ATG'), ('stop TAA', 'TAA'), ('stop TAG', 'TAG'), ('stop TGA', 'TGA'),
                    ('Shine-Dalgarno', 'AGGAG')]:
    frac = np.array([np.mean([motif in k for k in md_.top_kmers[c][:50]]) for c in range(hm.hidden_size)])
    hits = np.flatnonzero(frac >= 0.8)
    print(f'{name:>15}: {len(hits)} channels (>=80% of top-50 k-mers contain {motif}) e.g. {hits[:8].tolist()}')
RESULTS['motifs_top'] = [{'channel': int(c), 'ic': float(ics[c]), 'consensus': md_.consensus(c),
                          'top': md_.top_kmers[c][:20]} for c in order[:200]]

## 8. Why does Evo 2 think this BRCA1 mutation is harmful? (P5)

Real BRCA1 variants with lab-measured effects (Findlay et al. 2018, the data behind the Evo 2 paper's BRCA1 result).
For one loss-of-function (LOF) and one functional (FUNC) variant:
1. the zero-shot score (mean log-likelihood of the mutated window minus the normal one);
2. **patching**: copy each component from the mutated run into the normal run, and see how much of the mutation's
   effect on the following 200 letters it carries (total effect);
3. the direct-effect split for the letter right after the mutation.

Patching caches every component for an 8,192-letter window (~8 GB of CPU RAM), so High-RAM is needed.

In [ ]:
import gzip
for f in ['GRCh37.p13_chr17.fna.gz', '41586_2018_461_MOESM3_ESM.xlsx']:
    if not os.path.exists(f'data/{f}'):
        urllib.request.urlretrieve(f'{EVO2_RAW}/brca1/{f}', f'data/{f}')
with gzip.open('data/GRCh37.p13_chr17.fna.gz', 'rt') as h:
    chr17 = str(next(SeqIO.parse(h, 'fasta')).seq).upper()
brca1 = pd.read_excel('data/41586_2018_461_MOESM3_ESM.xlsx', header=2)[
    ['chromosome', 'position (hg19)', 'reference', 'alt', 'function.score.mean', 'func.class']]
print(brca1['func.class'].value_counts())
brca1.head()

In [ ]:
WINDOW = 8192
picks = {'LOF': brca1[brca1['func.class'] == 'LOF'].iloc[0], 'FUNC': brca1[brca1['func.class'] == 'FUNC'].iloc[0]}
RESULTS['brca1'] = {}
for label, row in picks.items():
    pos = int(row['position (hg19)']) - 1            # 0-based
    v = mh.make_variant(chr17, pos, row['reference'], row['alt'], window=WINDOW)
    dl = mh.delta_logp(hm, v)
    print(f"\n===== {label}: chr17:{pos+1} {row['reference']}>{row['alt']}  lab score={row['function.score.mean']:.2f}"
          f"  Evo 2 delta-loglik={dl:+.5f}")
    sweep = mh.explain_variant(hm, v, span=200)
    sweep.show(k=10)
    groups = {r.component: r.fraction for r in sweep.results if isinstance(r.component, str)}
    nxt = v.index + 1
    dref = mh.decompose_prediction(hm, v.ref_window, nxt, target=v.ref_window[nxt]).by_kind()
    dalt = mh.decompose_prediction(hm, v.alt_window, nxt, target=v.ref_window[nxt]).by_kind()
    print('direct effect on next letter, alt - ref:', {k: round(dalt[k] - dref[k], 3) for k in dref})
    RESULTS['brca1'][label] = {'pos': pos + 1, 'ref': row['reference'], 'alt': row['alt'], 'delta_logp': dl,
                               'group_restoration': groups}
    plt.figure(figsize=(6, 2.5))
    plt.bar(list(groups), list(groups.values()))
    plt.title(f'{label}: share of downstream effect carried (patching)'); plt.xticks(rotation=20); plt.show()

## 9. OPTIONAL: SAE features by operator type

The public Goodfire SAE reads block 26 of **`evo2_7b_262k`** (a different checkpoint), so this section swaps models.
For the strongest features on a stretch of E. coli DNA, it shows which operator types build them (an exact direct-effect
split of the SAE encoder input).

In [ ]:
RUN_SAE = False
if RUN_SAE:
    import gc
    from marv_hyena import sae as sae_mod
    del hm; gc.collect(); torch.cuda.empty_cache()
    hm = mh.HyenaModel.load(sae_mod.SAE_MODEL)
    sae = sae_mod.load_goodfire_sae(device=hm.device)
    s2 = genome[100_000:102_048]
    acts = sae_mod.feature_acts(hm, sae, s2)
    top_feats = acts.max(0).values.topk(5).indices.tolist()
    RESULTS['sae'] = []
    for f in top_feats:
        pos = int(acts[:, f].argmax())
        dec = sae_mod.decompose_feature(hm, sae, s2, pos, f)
        print(f'feature {f} @ {pos}: actual={dec.actual:+.3f}',
              {k: round(v, 3) for k, v in sorted(dec.by_kind().items())})
        RESULTS['sae'].append({'feature': f, 'position': pos, 'by_kind': dec.by_kind()})

## 10. Save results

This writes `results.json` (to Drive if mounted, otherwise download it from the file browser). Then append the outcomes
under each prediction in `PREDICTIONS.md`, without editing the predictions themselves.

In [ ]:
import json
out = '/content/drive/MyDrive/marv_hyena_results.json' if USE_DRIVE else 'results.json'
with open(out, 'w') as f:
    json.dump(RESULTS, f, indent=1, default=float)
print('wrote', out)